In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
#from sklearn.tree import DecisionTreeClassifier
#from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, classification_report, confusion_matrix

import random
import math

from sklearn.metrics import make_scorer, recall_score

In [2]:
# Load data
data = pd.read_csv("ACME-HappinessSurvey2020.csv")

# Separate features and target
X = data.drop("Y", axis=1)
y = data["Y"]

In [3]:
test_split_seed = 1234 #random.randrange(1000,9999)
#print(test_split_seed)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=test_split_seed
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
from sklearn.model_selection import GridSearchCV

dt = PassiveAggressiveClassifier(random_state=5678)

param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'class_weight': ['balanced', None],
    'loss': ['squared_hinge', 'hinge'],
    'early_stopping': [True, False],
    'average': [True, False]
}
recall_cl0_scr = make_scorer(recall_score, pos_label=0)

grid = GridSearchCV(
    dt,
    param_grid,
    scoring= recall_cl0_scr, #'recall',
    cv=5
)

grid.fit(X_train_scaled, y_train)

/data/Apziva/venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/data/Apziva/venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
/data/Apziva/venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, cate

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",PassiveAggres...om_state=5678)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'average': [True, False], 'class_weight': ['balanced', None], 'early_stopping': [True, False], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","make_scorer(r..., pos_label=0)"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation ti

In [5]:
# Make predictions
y_pred = grid.predict(X_test_scaled)

# Evaluate
print("recall:", recall_score(y_test, y_pred, pos_label=0))
#recall.append(recall_score(y_test, y_pred, pos_label=0))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
#print(test_split_seed)

recall: 0.42857142857142855

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.43      0.50        14
           1       0.50      0.67      0.57        12

    accuracy                           0.54        26
   macro avg       0.55      0.55      0.54        26
weighted avg       0.55      0.54      0.53        26


Confusion Matrix:
[[6 8]
 [4 8]]


In [7]:
grid.best_score_

np.float64(0.625)

In [6]:
#'''
#from sklearn.metrics import make_scorer, recall_score
#from sklearn.model_selection import GridSearchCV



## 1. Create a custom scorer targeting class 0
#recall_class0_scorer = make_scorer(recall_score, pos_label=0)



## 2. Pass it to the scoring argument of GridSearchCV
#grid_search = GridSearchCV(
#    estimator=your_model,
#    param_grid=your_param_grid,
#    scoring=recall_class0_scorer,
#    cv=5
#)



## 3. Fit the model
#grid_search.fit(X_train_scaled, y_train)
#'''